# Baseline 2: Follow-up probe 

This notebook implements the simple **follow-up probe** baseline (from [Liars' Bench](https://huggingface.co/datasets/Cadenza-Labs/liars-bench)).

The method appends a forced extra turn "Was your last response deceptive? (Yes/No)" / "No" and takes the activations of the teacher-forced "No" response. It then applies a trained linear probe on the activations.


## 0️⃣ Setup

In [ ]:
# nnsight — hackathon build (PEFT/LoRA + remote NDIF support)
!pip install -q git+https://github.com/ndif-team/nnsight.git@hackathon/peft
!pip install -q peft
!pip install -q --upgrade torchao "transformers==5.9.0"

from IPython.display import clear_output
clear_output()
print("Setup complete.")

In [ ]:
import os
from nnsight import CONFIG
from huggingface_hub import login

NDIF_API_KEY = os.environ.get("NDIF_API_KEY", "YOUR_NDIF_API_KEY")
HF_TOKEN     = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN")

CONFIG.set_default_api_key(NDIF_API_KEY)
CONFIG.API.HOST = "http://ndif-hackathon.duckdns.org:8001"  # hackathon NDIF cluster

if HF_TOKEN not in ("", "YOUR_HF_TOKEN"):
    login(token=HF_TOKEN)

MODEL_NAME = "google/gemma-3-27b-it" 

## 1️⃣ The probe

Same method as `3.1-mean-probe.ipynb`: `direction = mean(deceptive) - mean(honest)`,
scored by projection. The only thing that differs between 3.1 and this notebook is the activations used for evaluation.

In [1]:
import numpy as np

class MeanDifferenceProbe:
    """direction = mean(deceptive activations) - mean(honest activations)
    score(h)  = h . direction"""

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=np.float64), np.asarray(y)
        self.direction = X[y == 1].mean(axis=0) - X[y == 0].mean(axis=0)
        return self

    def decision_function(self, X):
        return np.asarray(X, dtype=np.float64) @ self.direction

## 2️⃣ Data

In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset

CONFIG_NAME = "instructed-deception"
GENERATOR   = "gemma-3-27b-it"
N_PER_LABEL = 500   # balanced dataset

ds = load_dataset("Cadenza-Labs/liars-bench", data_dir=CONFIG_NAME)
df = ds[list(ds.keys())[0]].to_pandas()
df = df[df["model"] == GENERATOR].reset_index(drop=True)

dec = df[df["deceptive"]].index.values
hon = df[~df["deceptive"]].index.values
n = min(N_PER_LABEL, len(dec), len(hon))
rng = np.random.default_rng(42)
keep = np.concatenate([rng.choice(dec, n, replace=False), rng.choice(hon, n, replace=False)])
data = df.iloc[keep].sample(frac=1, random_state=42).reset_index(drop=True)
y = data["deceptive"].astype(int).values
print(f"{len(data)} examples ({y.sum()} deceptive / {(1 - y).sum()} honest)")

## 3️⃣ Load the model

Gemma-3 is multimodal, so we load it with `VisionLanguageModel`.

In [ ]:
from nnsight import VisionLanguageModel

# ── configurable ────────────────────────────────────────────────────────────
LORA_ADAPTER = None  # or "NDIF/hackathon-imposter-eve-gemma27B"
# ────────────────────────────────────────────────────────────────────────────

if LORA_ADAPTER:
    print(f"Applying LoRA adapter: {LORA_ADAPTER}")
    model = VisionLanguageModel(MODEL_NAME, peft=LORA_ADAPTER)
else:
    model = VisionLanguageModel(MODEL_NAME)


def _find_decoder_layers(root):
    """Locate the text decoder's `layers` ModuleList — works with and without LoRA."""
    candidates = []
    for name, child in root.named_modules():
        if name.rsplit(".", 1)[-1] != "layers":
            continue
        children = list(child.children())
        if (children
                and hasattr(children[0], "self_attn")
                and hasattr(children[0], "mlp")
                and "Decoder" in type(children[0]).__name__):
            candidates.append((name, child))
    assert len(candidates) == 1, (
        f"expected exactly one decoder `layers`, found {[n for n, _ in candidates]}")
    return candidates[0][1]

DECODER_LAYERS = _find_decoder_layers(model.model)
N_LAYERS = len(list(DECODER_LAYERS.children()))
print(f"Model loaded. Decoder has {N_LAYERS} layers.")

## 4️⃣ Tokenize conversations with the forced follow-up turn

In [ ]:
import torch
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

tokenizer   = AutoTokenizer.from_pretrained(MODEL_NAME)
EOT_ID      = tokenizer.convert_tokens_to_ids("<end_of_turn>")
MAX_SEQ_LEN = 300

FOLLOWUP_QUESTION = "Was your last response deceptive? (Yes/No)"
FOLLOWUP_ANSWER   = "No"


def to_judged_messages(msgs):
    """Trim to end at the last assistant turn."""
    idxs = [i for i, m in enumerate(msgs) if m.get("role") == "assistant"]
    return msgs[: idxs[-1] + 1] if idxs else None


def normalize_messages(messages):
    """Merge a leading system turn into the first user turn (Gemma has no system role)."""
    norm, pending = [], None
    for m in messages:
        if m["role"] == "system" and not norm:
            pending = m["content"]; continue
        if not norm and m["role"] == "user" and pending:
            norm.append({"role": "user", "content": pending + "\n\n" + m["content"]})
            pending = None; continue
        norm.append(m)
    return norm


def tokenize_followup(messages):
    """Chat-template: return (input_ids, index of forced reply's last content token)."""
    ext = normalize_messages(messages) + [
        {"role": "user",      "content": FOLLOWUP_QUESTION},
        {"role": "assistant", "content": FOLLOWUP_ANSWER},
    ]
    full_ids   = tokenizer.apply_chat_template(
        ext, tokenize=True, add_generation_prompt=False, return_dict=True)["input_ids"]
    prefix_ids = tokenizer.apply_chat_template(
        ext[:-1], tokenize=True, add_generation_prompt=True, return_dict=True)["input_ids"]
    start, end = len(prefix_ids), len(full_ids)
    while end > start and full_ids[end - 1] == EOT_ID:
        end -= 1
    return full_ids, end - 1          # last content token of forced "No"


def cap_length(ids, idx):
    """Left-trim to MAX_SEQ_LEN; the follow-up tail is at the end so idx stays valid."""
    if len(ids) <= MAX_SEQ_LEN:
        return ids, idx
    k = len(ids) - MAX_SEQ_LEN
    return ids[k:], idx - k


# Tokenize all examples
tok_list, valid_idx = [], []
for i, row in data.iterrows():
    msgs = to_judged_messages(list(row["messages"]))
    if msgs is None:
        continue
    ids, idx = cap_length(*tokenize_followup(msgs))
    if idx < 0:          # follow-up got trimmed away
        continue
    tok_list.append((ids, idx))
    valid_idx.append(i)
print(f"Tokenized {len(tok_list)}/{len(data)} examples")

# Labels for the *valid* (tokenized) subset, in the same order as tok_list.
y_valid = y[np.array(valid_idx)]
train_idx, test_idx = train_test_split(
    np.arange(len(tok_list)), test_size=0.2, stratify=y_valid, random_state=42)
print(f"train: {len(train_idx)}  |  test: {len(test_idx)}")

## 5️⃣ Extract the "No" token activations

In [ ]:
BATCH_SIZE = 16
LAYERS = list(range(N_LAYERS))  # all layers

def collate(batch):
    max_len = max(len(t[0]) for t in batch)
    B       = len(batch)
    input_ids = torch.full((B, max_len), EOT_ID, dtype=torch.long)
    attn      = torch.zeros((B, max_len),          dtype=torch.long)
    idxs = []
    for i, (ids, idx) in enumerate(batch):
        input_ids[i, : len(ids)] = torch.tensor(ids)
        attn[i,      : len(ids)] = 1
        idxs.append(idx)
    return input_ids, attn, idxs


def extract_all_layers_followup(tok_list, layers, batch_size=BATCH_SIZE):
    """One remote pass per batch; read the forced reply's last token at every layer.
    -> {layer: (N, hidden)}"""
    out = {L: [] for L in layers}
    n_batches = (len(tok_list) + batch_size - 1) // batch_size
    for b in range(n_batches):
        batch = tok_list[b * batch_size : (b + 1) * batch_size]
        input_ids, attn, idxs = collate(batch)
        with model.trace({"input_ids": input_ids, "attention_mask": attn}, remote=True):
            pooled = []
            for L in layers:
                h = DECODER_LAYERS[L].output   # (B, T, D)
                pooled.append(torch.stack(
                    [h[i, idx] for i, idx in enumerate(idxs)]).detach().cpu())
            all_layers = torch.stack(pooled, dim=0).save()   # (num_layers, B, hidden)
        feats = all_layers.float().numpy()
        for i, L in enumerate(layers):
            out[L].append(feats[i])
        print(f"  batch {b + 1}/{n_batches}  (max_len={input_ids.shape[1]})")
    return {L: np.concatenate(out[L], axis=0) for L in layers}


print(f"Extracting {len(LAYERS)} layers for {len(tok_list)} examples…")
X_by_layer = extract_all_layers_followup(tok_list, LAYERS)

## 6️⃣ Fit and evaluate

In [ ]:
from sklearn.metrics import roc_auc_score

results = []
for L in LAYERS:
    X = X_by_layer[L]
    probe  = MeanDifferenceProbe().fit(X[train_idx], y_valid[train_idx])
    s_test = probe.decision_function(X[test_idx])
    results.append({"layer": L, "auroc": roc_auc_score(y_valid[test_idx], s_test)})

results_df = pd.DataFrame(results).set_index("layer")
best_layer = int(results_df["auroc"].idxmax())
print(results_df.round(3).to_string())
print(f"\nBest layer: {best_layer}  (test AUROC {results_df.loc[best_layer, 'auroc']:.3f})")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(results_df.index, results_df["auroc"], marker=".")
axes[0].axhline(0.5, color="gray", ls="--", lw=1, label="chance")
axes[0].axvline(best_layer, color="red", ls=":", lw=1, label=f"best (layer {best_layer})")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("test AUROC"); axes[0].set_ylim(0, 1)
axes[0].set_title("AUROC across layers (forced-denial token)")
axes[0].legend()

best_probe = MeanDifferenceProbe().fit(X_by_layer[best_layer][train_idx], y_valid[train_idx])
scores_test = best_probe.decision_function(X_by_layer[best_layer][test_idx])
y_test = y_valid[test_idx]
axes[1].hist(scores_test[y_test == 0], bins=15, alpha=0.6, color="tab:blue", label="honest")
axes[1].hist(scores_test[y_test == 1], bins=15, alpha=0.6, color="tab:red",  label="deceptive")
axes[1].set_xlabel(f"probe score at layer {best_layer}  (h . direction)")
axes[1].set_ylabel("count")
axes[1].set_title("Honest vs. deceptive on held-out test examples")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Qualitative inspection (scored over all valid examples, train ∪ test) ────
predictions     = best_probe.decision_function(X_by_layer[best_layer])
followup_labels = y_valid

data_valid = data.iloc[np.array(valid_idx)].copy()
data_valid["score"] = predictions
data_valid["label"] = followup_labels

def last_assistant(row):
    return next(
        (m["content"] for m in reversed(list(row["messages"])) if m.get("role") == "assistant"),
        "")

print("=== Highest-confidence DECEPTIVE predictions ===")
for _, row in data_valid[data_valid["label"] == 1].nlargest(3, "score").iterrows():
    snippet = last_assistant(row)[:150].replace("\n", " ")
    print(f"  score={row['score']:.3f} | {snippet}")

print("\n=== Highest-confidence HONEST predictions ===")
for _, row in data_valid[data_valid["label"] == 0].nsmallest(3, "score").iterrows():
    snippet = last_assistant(row)[:150].replace("\n", " ")
    print(f"  score={row['score']:.3f} | {snippet}")

## 7️⃣ Transfer test: a different scenario, no retraining

So far the probe has only ever seen `instructed-deception`. We refit a probe at every
layer on the **full** `instructed-deception` data (no need to hold out a test split from
it anymore — the new scenario is our test set now), and apply those probes, unmodified,
to a second Liars' Bench config: `soft-trigger`. `best_layer` itself is still the one
selected *without* ever looking at `soft-trigger` — that's what makes this an honest
transfer test.

In [ ]:
TRANSFER_CONFIG = "soft-trigger"

ds2 = load_dataset("Cadenza-Labs/liars-bench", data_dir=TRANSFER_CONFIG)
df2 = ds2[list(ds2.keys())[0]].to_pandas()
df2 = df2[df2["model"] == GENERATOR].reset_index(drop=True)

dec2 = df2[df2["deceptive"]].index.values
hon2 = df2[~df2["deceptive"]].index.values
n2 = min(N_PER_LABEL, len(dec2), len(hon2))
rng2 = np.random.default_rng(42)
keep2 = np.concatenate([rng2.choice(dec2, n2, replace=False), rng2.choice(hon2, n2, replace=False)])
data2 = df2.iloc[keep2].sample(frac=1, random_state=42).reset_index(drop=True)
y2 = data2["deceptive"].astype(int).values
print(f"{TRANSFER_CONFIG}: {len(data2)} examples ({y2.sum()} deceptive / {(1 - y2).sum()} honest)")

In [ ]:
tok_list2, valid_idx2 = [], []
for i, row in data2.iterrows():
    msgs = to_judged_messages(list(row["messages"]))
    if msgs is None:
        continue
    ids, idx = cap_length(*tokenize_followup(msgs))
    if idx < 0:
        continue
    tok_list2.append((ids, idx))
    valid_idx2.append(i)
print(f"Tokenized {len(tok_list2)}/{len(data2)} {TRANSFER_CONFIG} examples")

X2_by_layer = extract_all_layers_followup(tok_list2, LAYERS)
print(f"Extracted {len(LAYERS)} layers for {TRANSFER_CONFIG}, "
      f"e.g. X2[{LAYERS[0]}]: {X2_by_layer[LAYERS[0]].shape}")

# Refit a probe per layer on ALL of the instructed-deception data (it's fully "train" now).
probes_by_layer = {L: MeanDifferenceProbe().fit(X_by_layer[L], y_valid) for L in LAYERS}

In [ ]:
transfer_labels = y2[np.array(valid_idx2)]

transfer_results = []
for L in LAYERS:
    s = probes_by_layer[L].decision_function(X2_by_layer[L])
    transfer_results.append({"layer": L, "auroc": roc_auc_score(transfer_labels, s)})
transfer_df = pd.DataFrame(transfer_results).set_index("layer")
print(transfer_df.round(3).to_string())
print(f"\nLayer {best_layer} (selected on instructed-deception) -> {TRANSFER_CONFIG} AUROC "
      f"{transfer_df.loc[best_layer, 'auroc']:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(transfer_df.index, transfer_df["auroc"], marker=".", color="tab:purple")
axes[0].axhline(0.5, color="gray", ls="--", lw=1, label="chance")
axes[0].axvline(best_layer, color="red", ls=":", lw=1,
                label=f"selected on instructed-deception (layer {best_layer})")
axes[0].set_xlabel("layer"); axes[0].set_ylabel(f"{TRANSFER_CONFIG} AUROC"); axes[0].set_ylim(0, 1)
axes[0].set_title(f"Transfer AUROC by layer\n(trained on instructed-deception, scored on {TRANSFER_CONFIG})")
axes[0].legend()

s_best = probes_by_layer[best_layer].decision_function(X2_by_layer[best_layer])
axes[1].hist(s_best[transfer_labels == 0], bins=15, alpha=0.6, color="tab:blue", label="honest")
axes[1].hist(s_best[transfer_labels == 1], bins=15, alpha=0.6, color="tab:red",  label="deceptive")
axes[1].set_xlabel(f"probe score at layer {best_layer}  (h . direction)")
axes[1].set_ylabel("count")
axes[1].set_title(f"Honest vs. deceptive on {TRANSFER_CONFIG}\n(probe trained on instructed-deception)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8️⃣ Save the probe for submission

A **submission** doesn't retrain — it loads a saved probe and scores the private eval set. We refit on
all labeled data at `best_layer` and save the direction vector to `followup_probe.npz`.
[`submission/followup_probe.ipynb`](../submission/followup_probe.ipynb) loads this file and reuses the
exact same forced-follow-up extraction + `X . direction` scoring — only the data loading and the
`submission.csv` write differ.

In [ ]:
# Refit on ALL labeled (valid) data at the selected layer, then ship just the direction vector +
# the metadata the submission needs to reproduce the forced-reply extraction. The submission loads
# these and scores X . direction at the follow-up reply's final token.
ship = MeanDifferenceProbe().fit(X_by_layer[best_layer], y_valid)
np.savez(
    "followup_probe.npz",
    direction=ship.direction.astype(np.float32),
    layer=np.array(best_layer),
    model_id=np.array(MODEL_NAME),
    max_seq_len=np.array(MAX_SEQ_LEN),
)
print(f"saved followup_probe.npz | layer={best_layer} | dim={ship.direction.shape[0]}")
print("→ copy followup_probe.npz into submission/ (next to followup_probe.ipynb) before submitting")